> **Release dependency:** the original `clinical_review/` helper package was absent from the supplied source tree. This notebook is retained for documentation but cannot run until those original helpers are restored. See `docs/REPRODUCIBILITY.md`. No completed reviewer data are distributed.

# BR08AS — Import Shaoxing offline ratings

Run after BR08S and completion/upload of the human rating form. Uses the source identity pinned before review, preserving the original native import and adjudication checks. No model inference.


In [ ]:
from pathlib import Path
import sys
import json
import hashlib
import os
from datetime import datetime, timezone

EXP = Path('/path/to/repository/experiment02')
REVIEW_DIR = Path(
    '/path/to/repository/outputs/real_run_001/'
    'experiment02_v1/context_fit/v1/shaoxing/clinical_review/shaoxing/calibration_v1'
)
INPUT_PATH = REVIEW_DIR / 'PUBLIC/report_ratings_completed.json'
if not (EXP / 'common.py').is_file():
    raise FileNotFoundError('Set EXP to experiment02 containing common.py')
sys.path.insert(0, str(EXP))
from common import read, sha, signature, immutable, lock
from clinical_review.workflow import (
    _check, RATING_FIELDS, validate_rating, needs_adjudication,
    save_rating, prepare_adjudication, progress
)
if not INPUT_PATH.is_file():
    raise FileNotFoundError('Upload the completed rating JSON to: ' + str(INPUT_PATH))
print('Kernel:', sys.executable)
print('Review package:', REVIEW_DIR)
print('Upload:', INPUT_PATH)
print('Current progress:', progress(REVIEW_DIR))


## 1. Validate the whole batch before writing

The checks below require all 40 report IDs exactly once, the registered reviewer, completed typed fields, and the native consistency rules. They also reject a different report/image package and any attempt to overwrite conflicting existing or frozen ratings. No report text, model identity or clinical interpretation is printed.

The identity constants are calculated from the downloaded PUBLIC files, not invented case/report associations. Do not replace them merely to bypass an identity mismatch. If you intentionally create a new review package, create a corresponding new import notebook/version.

In [ ]:
# Pinned by BR08S BEFORE the reviewer edits the form; never recompute to bypass mismatches.
source_identity = read(REVIEW_DIR / 'PRIVATE/offline_rating_source.json')
if source_identity['package_protocol_sha256'] != sha(REVIEW_DIR / 'PRIVATE/protocol.json'):
    raise ValueError('Review package changed since form creation')
EXPECTED_REPORTS_SIGNATURE = source_identity['reports_signature']
EXPECTED_IMAGE_HASHES = source_identity['image_hashes']

def validate_completed_ratings():
    protocol = _check(REVIEW_DIR)
    required = ('PRIVATE/references_submitted.json', 'PUBLIC/reports.json')
    for name in required:
        if not (REVIEW_DIR / name).is_file():
            raise ValueError('Finish BR08 reference submission/report release first: ' + name)
    reports = read(REVIEW_DIR / 'PUBLIC/reports.json')
    if signature(reports) != EXPECTED_REPORTS_SIGNATURE:
        raise ValueError('Reports differ from the downloaded blinded package used for this notebook')
    for relative, expected_sha in EXPECTED_IMAGE_HASHES.items():
        if sha(REVIEW_DIR / 'PUBLIC' / relative) != expected_sha:
            raise ValueError('Reviewed ECG image mismatch: ' + relative)
    payload = INPUT_PATH.read_bytes()
    source_sha = hashlib.sha256(payload).hexdigest()
    # Reject duplicate keys and nonstandard JSON constants instead of silently accepting them.
    def unique_keys(pairs):
        result = {}
        for key, value in pairs:
            if key in result:
                raise ValueError('Duplicate JSON key: ' + key)
            result[key] = value
        return result
    def invalid_constant(value):
        raise ValueError('Nonstandard JSON number: ' + value)
    data = json.loads(payload, object_pairs_hook=unique_keys, parse_constant=invalid_constant)
    if not isinstance(data, dict) or data.get('schema_version') != 'blinded_report_ratings_v1':
        raise ValueError('Unexpected rating file schema')
    if data.get('stage') != protocol.get('stage'):
        raise ValueError('Rating stage differs from the native package')
    if data.get('source_reports_file') != 'reports.json':
        raise ValueError('Unexpected source report file')
    rows = data.get('ratings')
    if not isinstance(rows, list) or not rows or not all(isinstance(r, dict) for r in rows):
        raise ValueError('ratings must be a nonempty list of rating objects')
    expected = {r['review_id']: r for r in reports}
    ids = [r.get('review_id') for r in rows]
    if (any(not isinstance(rid, str) for rid in ids)
            or len(set(ids)) != len(ids) or set(ids) != set(expected)):
        raise ValueError('Rating IDs must match every released report exactly once')
    if type(data.get('expected_reports')) is not int or data['expected_reports'] != len(reports):
        raise ValueError('expected_reports does not match this package')
    snapshot = REVIEW_DIR / 'PRIVATE/imported_report_ratings.json'
    if snapshot.exists() and sha(snapshot) != source_sha:
        raise ValueError('A different rating upload was already imported; preserve the original')
    frozen_path = REVIEW_DIR / 'PRIVATE/initial_submitted.json'
    frozen = None
    if frozen_path.exists():
        frozen_rows = read(frozen_path)
        frozen = {r['review_id']: r for r in frozen_rows}
        if len(frozen) != len(frozen_rows) or set(frozen) != set(expected):
            raise ValueError('Frozen initial rating identifiers differ')
    existing = set()
    records = []
    for row in rows:
        rid = row['review_id']
        if set(row) != set(RATING_FIELDS) | {'review_id'}:
            raise ValueError('Unexpected or missing rating fields: ' + rid)
        rating = {k: row[k] for k in RATING_FIELDS}
        try:
            validate_rating(rating, protocol['reviewer_id'])
        except (ValueError, TypeError) as exc:
            raise ValueError(rid + ': ' + str(exc)) from exc
        draft = REVIEW_DIR / 'PUBLIC/rating_drafts' / (rid + '.json')
        if draft.exists():
            saved = read(draft)
            if (saved.get('review_id') != rid
                    or {k: saved.get(k) for k in RATING_FIELDS} != rating):
                raise ValueError('Conflicting existing draft; no overwrite: ' + rid)
            existing.add(rid)
        if frozen is not None:
            saved = frozen[rid]
            if {k: saved.get(k) for k in RATING_FIELDS} != rating:
                raise ValueError('Conflicting frozen initial rating; use adjudication: ' + rid)
        records.append(dict(review_id=rid, **rating))
    return protocol, records, existing, payload, source_sha, frozen is not None

protocol, records, existing, payload, source_sha, already_frozen = validate_completed_ratings()
print('VALID:', len(records), 'completed ratings')
print('Identical existing drafts:', len(existing))
print('Initial ratings already frozen:', already_frozen)
print('Rating-triggered adjudications:', sum(needs_adjudication(r) for r in records))
print('An uninterpretable independent reference can also trigger adjudication.')


## 2. Import the initial ratings

Run this cell to import. It revalidates under a single-writer lock, keeps an exact private copy of the uploaded JSON, and calls the native `save_rating` function for each unsaved entry. Matching drafts are skipped, so an interruption can be resumed using the same file. A byte-different upload after import stops rather than silently replacing the audit source.

Do not run BR08 editing cells in parallel. Once initial ratings have been frozen, an identical rerun is verification only; no frozen rating is rewritten.

In [ ]:
def import_completed_ratings():
    with lock(REVIEW_DIR / 'report_rating_import.lock'):
        protocol, records, existing, payload, source_sha, frozen = validate_completed_ratings()
        snapshot = REVIEW_DIR / 'PRIVATE/imported_report_ratings.json'
        if not snapshot.exists():
            temporary = snapshot.with_name(snapshot.name + '.tmp')
            with temporary.open('wb') as stream:
                stream.write(payload)
                stream.flush()
                os.fsync(stream.fileno())
            temporary.replace(snapshot)
        imported = 0
        if not frozen:
            for row in records:
                if row['review_id'] in existing:
                    continue
                save_rating(REVIEW_DIR, row['review_id'],
                            **{k: row[k] for k in RATING_FIELDS})
                imported += 1
        log_path = REVIEW_DIR / 'PRIVATE/report_rating_notebook_import.json'
        timestamp = (read(log_path)['imported_at_utc'] if log_path.exists()
                     else datetime.now(timezone.utc).isoformat())
        immutable(log_path, dict(
            workflow='BR08AS_import_completed_report_ratings_v1', imported_at_utc=timestamp,
            source_sha256=source_sha, review_ids=[r['review_id'] for r in records],
            reviewer_id=protocol['reviewer_id'],
            package_protocol_sha256=sha(REVIEW_DIR / 'PRIVATE/protocol.json'),
            reports_signature=EXPECTED_REPORTS_SIGNATURE,
            image_hashes=EXPECTED_IMAGE_HASHES, protected_assets_modified=False))
    return dict(new_ratings_imported=imported, identical_existing_drafts=len(existing),
                initial_ratings_already_frozen=frozen, progress=progress(REVIEW_DIR))

print(import_completed_ratings())


## 3. Freeze the imported initial ratings and prepare adjudication

After reviewing the validation/import summaries, run this cell. It freezes all initial ratings and prepares the queue using the existing BR08 rules. The same clinician must perform a separate second assessment of queued items, preferably in a later session. The initial ratings and independent interpretations remain intact.

The queue may contain many reports; that is not an import error. Retain a disagreement if supported by the ECG. Do not revise judgments to improve study results. An identical rerun resumes the existing queue.

In [ ]:
with lock(REVIEW_DIR / 'report_rating_import.lock'):
    validate_completed_ratings()
    log = REVIEW_DIR / 'PRIVATE/report_rating_notebook_import.json'
    if not log.is_file():
        raise ValueError('Run the import cell before freezing ratings')
    adjudication_status = prepare_adjudication(REVIEW_DIR)
print(adjudication_status)
print('Progress:', progress(REVIEW_DIR))
print('Initial ratings:', REVIEW_DIR / 'PRIVATE/initial_submitted.json')
print('Clinician queue:', REVIEW_DIR / 'PUBLIC/adjudication_queue.json')
print('Continue in BR08 at the SAME-REVIEWER ADJUDICATION section using this REVIEW_DIR.')
print('Do not re-enter initial ratings or edit frozen interpretations.')
print('Run BR09 only after every required adjudication has been saved.')


## What happens next

Use BR08's adjudication section to record **retain_initial**, **revise_initial**, or **unresolved**, with the clinician's reason and evidence reviewed. This notebook deliberately makes no adjudication decisions and does not generate clinical scores. If offline adjudication is preferred, the queue can be downloaded; a separate validated offline adjudication import is not implemented by BR08AS.

After all required adjudications are complete, use BR09 for analysis. Ten calibration ECGs do not establish definitive clinical efficacy or inter-rater reliability.